# NewFace Colab Runner

- Runs on free Colab.
- Downloads the required models directly from your GitHub Release.
- Does not depend on Google Drive.
- Launches the default Web UI once the model set is ready.


In [ ]:
REPO_URL = "https://github.com/tjwcold/newface.git"
REPO_DIR = "/content/newface"
MODEL_REPO = "tjwcold/newface"
MODEL_RELEASE_TAG = "videokit-models"
DEFAULT_PROCESSOR = "face_swapper"
DEFAULT_FACE_SWAPPER_MODEL = "hyperswap_1a_256"

REQUIRED_NOW = sorted([
    "fairface.hash",
    "fairface.onnx",
    "yoloface_8n.hash",
    "yoloface_8n.onnx",
    "2dfan4.hash",
    "2dfan4.onnx",
    "xseg_1.hash",
    "xseg_1.onnx",
    "bisenet_resnet_34.hash",
    "bisenet_resnet_34.onnx",
    "arcface_w600k_r50.hash",
    "arcface_w600k_r50.onnx",
    "kim_vocal_2.hash",
    "kim_vocal_2.onnx",
    "hyperswap_1a_256.hash",
    "hyperswap_1a_256.onnx",
])

MODEL_BASE_URL = f"https://github.com/{MODEL_REPO}/releases/download/{MODEL_RELEASE_TAG}"

print("repo:", REPO_URL)
print("model repo:", MODEL_REPO)
print("model release tag:", MODEL_RELEASE_TAG)
print("default processor:", DEFAULT_PROCESSOR)
print("default face swapper model:", DEFAULT_FACE_SWAPPER_MODEL)


In [ ]:
import os

os.chdir("/content")
print(os.getcwd())
!rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(os.getcwd())
!apt-get -y update
!apt-get -y install ffmpeg
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install -r requirements.txt


In [ ]:
import urllib.request
from pathlib import Path

repo_dir = Path(REPO_DIR)
model_dir = repo_dir / ".assets" / "models"
model_dir.mkdir(parents=True, exist_ok=True)

missing = []
downloaded = []

for name in REQUIRED_NOW:
    target = model_dir / name
    url = f"{MODEL_BASE_URL}/{name}"
    if target.exists():
        downloaded.append(name)
        continue
    try:
        urllib.request.urlretrieve(url, target)
        downloaded.append(name)
        print("downloaded", name)
    except Exception as exc:
        missing.append(name)
        print("missing", name, exc)

missing_file = repo_dir / "colab_missing_models.txt"
missing_file.write_text("\n".join(missing), encoding="utf-8")

print("\nready:", len(downloaded))
print("missing:", len(missing))
print("missing list:", missing_file)


In [ ]:
from pathlib import Path

launch_files = [
    Path(REPO_DIR) / "newface/uis/layouts/default.py",
    Path(REPO_DIR) / "newface/uis/layouts/jobs.py",
    Path(REPO_DIR) / "newface/uis/layouts/webcam.py",
]

old = "ui.launch(favicon_path = 'newface.ico', inbrowser = state_manager.get_item('open_browser'))"
new = "ui.launch(favicon_path = 'newface.ico', inbrowser = state_manager.get_item('open_browser'), share = True, server_name = '0.0.0.0')"

for path in launch_files:
    text = path.read_text(encoding="utf-8")
    if old in text and "share = True" not in text:
        path.write_text(text.replace(old, new), encoding="utf-8")
        print("patched", path.name)
    else:
        print("ok", path.name)


In [ ]:
if missing:
    raise SystemExit(
        "missing required models in your GitHub Release. check /content/newface/colab_missing_models.txt and upload those files first."
    )

print("default startup model set is ready.")


In [ ]:
import os
os.chdir(REPO_DIR)
print(os.getcwd())
!python newface.py run --execution-providers cuda --face-detector-model yolo_face --face-landmarker-model 2dfan4 --face-occluder-model xseg_1 --face-parser-model bisenet_resnet_34 --voice-extractor-model kim_vocal_2 --processors face_swapper --face-swapper-model hyperswap_1a_256 --log-level info
